# Exploratory data analysis: SMART-Z (2017–2018)

Report Section IV-D. Everything below uses the **training split of seed 0 only** (no validation/test data informs a design choice). All numbers are computed from the real feature store built by `scripts/run_pipeline.py --stage features`.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
from IPython.display import Image, display
from tcse.utils import load_config
from tcse.eda import EDA
pd.set_option('display.width', 200)
cfg = load_config('../configs/default.yaml')
eda = EDA(cfg, 'smartz')
fig = lambda n: display(Image(filename=f'../results/figures/eda_smartz_{n}.png'))
print('attributes present in training data:', eda.attrs)

## 1. Summary statistics and class ratio per horizon

In [ ]:
display(eda.summary_stats())
display(eda.class_ratio())

## 2. Missing-data patterns (raw values, before forward fill)

In [ ]:
miss = eda.missingness(); fig('missingness')

## 3. Failed vs. healthy drives: box plots and Kolmogorov–Smirnov tests

In [ ]:
ks = eda.failed_vs_healthy(); display(ks); fig('boxplots_ks')

## 4. Attribute correlations

In [ ]:
R, high = eda.correlation(); fig('correlation')
print('pairs with |r| > 0.95 (one of each is dropped by feature selection):')
for h in high: print(h)

## 5. Average 30-day pre-failure trajectories

In [ ]:
traj = eda.prefailure_trajectories(); fig('prefailure_trajectories')

## 6. Two-dimensional PCA projection

In [ ]:
ev = eda.pca(); print('explained variance ratio:', ev.round(4)); fig('pca')

## 7. Choice of the trend window m

KS separation (healthy vs. fails-within-7-days rows) of the delta / slope / std trend features for each candidate window, plus the window chosen on the validation set by `tcse.experiments.select_window` (TCSE validation cost).

In [ ]:
piv, summ = eda.window_separation(); display(summ); display(piv)
import json, pathlib
for f in pathlib.Path('../results/metrics/window_selection').glob('smartz*/chosen.json'): print(f.parent.name, json.loads(f.read_text()))